# Module 13 — Notebook 1: Confusion Matrix

## Learning Objectives

By the end of this notebook you will be able to:

- Build and interpret a confusion matrix for a binary classifier
- Explain why accuracy alone can be misleading for imbalanced datasets
- Compute accuracy from a confusion matrix dict

## Why This Matters for AI Research Engineering

In safety evaluation, understanding *where* a classifier fails is as important as knowing its overall score. A classifier with 90% accuracy might still miss most harmful outputs — if 90% of outputs are safe, a classifier that always says "safe" achieves 90% accuracy without catching a single real problem.

A confusion matrix breaks down predictions into four categories, letting you see exactly which kinds of errors your classifier makes.

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_contains, check_keys, check_length
print("Setup complete.")

In [ ]:
# Load the synthetic model outputs
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

print(f"Loaded {len(outputs)} records.")
print("Sample record:", outputs[0])

# Keyword triggers for classifier v1
TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']

def classify_v1(response):
    """Return True (flagged) if any trigger phrase appears in the response."""
    return any(trigger in response for trigger in TRIGGERS_V1)

# Apply the classifier to all outputs
predictions_v1 = [classify_v1(r['response']) for r in outputs]
ground_truth = [r['flagged'] for r in outputs]

print(f"\nClassifier v1 predictions: {predictions_v1}")
print(f"Ground truth labels:       {ground_truth}")

## Concept: The Confusion Matrix

A confusion matrix organises predictions into four buckets based on whether the prediction was correct:

|  | **Predicted: Flagged** | **Predicted: Not Flagged** |
|---|---|---|
| **Actually Flagged** | TP (True Positive) | FN (False Negative) |
| **Actually Not Flagged** | FP (False Positive) | TN (True Negative) |

- **TP (True Positive):** Correctly flagged a harmful output
- **FP (False Positive):** Flagged a safe output — a false alarm
- **FN (False Negative):** Missed a harmful output — the dangerous kind of error
- **TN (True Negative):** Correctly left a safe output unflagged

For safety classifiers, **FNs** are usually the most serious error — they represent harmful content that slips through.

In [ ]:
# Worked example: building a confusion dict from scratch
# We iterate over (prediction, label) pairs and count each cell.

example_preds  = [True, False, True, False, True]
example_labels = [True, True, False, False, True]

demo_confusion = {'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0}
for pred, label in zip(example_preds, example_labels):
    if pred and label:
        demo_confusion['tp'] += 1   # predicted flagged, actually flagged
    elif pred and not label:
        demo_confusion['fp'] += 1   # predicted flagged, actually safe
    elif not pred and label:
        demo_confusion['fn'] += 1   # predicted safe, actually flagged
    else:
        demo_confusion['tn'] += 1   # predicted safe, actually safe

print("Demo confusion matrix:", demo_confusion)
# Expected: {'tp': 2, 'fp': 1, 'fn': 1, 'tn': 1}

## Exercise 1 — Build the Confusion Matrix for Classifier V1

Using `predictions_v1` and `ground_truth`, build a confusion matrix dict:

```python
confusion_v1 = {'tp': ..., 'fp': ..., 'fn': ..., 'tn': ...}
```

Iterate over `zip(predictions_v1, ground_truth)` and count each cell, just like the worked example above.

In [ ]:
# Your code here
confusion_v1 = {'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0}

# TODO: iterate over zip(predictions_v1, ground_truth) and fill in each cell


In [ ]:
check_keys(confusion_v1, ['tp', 'fp', 'fn', 'tn'], "confusion_v1 has correct keys")
check_equal(confusion_v1['tp'], 5, "TP count")
check_equal(confusion_v1['fp'], 0, "FP count")
check_equal(confusion_v1['fn'], 2, "FN count")
check_equal(confusion_v1['tn'], 13, "TN count")

## Exercise 2 — Compute Accuracy

Accuracy is the fraction of all predictions that were correct:

```
accuracy = (TP + TN) / total
```

Compute `accuracy_v1` using `confusion_v1`. There are 20 outputs total. Round to 4 decimal places.

In [ ]:
# Your code here
accuracy_v1 = None  # replace with your calculation


In [ ]:
check_approx(accuracy_v1, 0.9, 0.001, "Classifier v1 accuracy")

## Concept: When Accuracy Is Misleading

Our dataset has **7 flagged outputs** and **13 safe outputs** out of 20 total — the classes are imbalanced.

A trivially simple classifier that **always predicts "not flagged"** would:
- Get every safe output right (13 TN)
- Get every flagged output wrong (7 FN)
- Score **13 / 20 = 65% accuracy** without ever catching a single problem

This is called the **majority-class baseline**. Any real classifier needs to beat this baseline by a meaningful margin to be useful.

Classifier v1 achieves 90% accuracy — 25 percentage points above the majority-class baseline. That gap tells us something real is being learned. But those 2 FNs still represent harmful outputs that escaped detection.

## Exercise 3 — Majority-Class Baseline

Build the confusion matrix for a classifier that **always predicts "not flagged"**, then compute its accuracy and compare it to classifier v1.

```python
majority_class_confusion = {'tp': 0, 'fp': 0, 'fn': 7, 'tn': 13}
majority_accuracy = round((0 + 13) / 20, 4)
accuracy_gap = round(accuracy_v1 - majority_accuracy, 4)
```

In [ ]:
# Your code here
majority_class_confusion = None  # replace with the dict
majority_accuracy = None         # replace with the calculation
accuracy_gap = None              # replace with the calculation


In [ ]:
check_approx(majority_accuracy, 0.65, 0.001, "Majority-class accuracy")
check_approx(accuracy_gap, 0.25, 0.001, "Accuracy gap (v1 vs majority)")

## Summary

| Metric | Classifier V1 | Majority-class baseline |
|---|---|---|
| TP | 5 | 0 |
| FP | 0 | 0 |
| FN | 2 | 7 |
| TN | 13 | 13 |
| Accuracy | 0.90 | 0.65 |

Key takeaways:

- A confusion matrix shows *where* errors occur, not just how many there are
- Accuracy alone is not enough when classes are imbalanced
- Always compare your classifier to a simple baseline
- Classifier v1 has **no false positives** but **2 false negatives** — it never raises false alarms, but it misses some harmful outputs

In the next notebook we will dig into those 2 false negatives and ask: *what do the failing examples have in common?*